# OmniCare Clinical Copilot - Notebook 1: Consultation & SOAP Note

**Multi-Agent Pipeline:** Patient Scenario --> Audio/Transcript Input --> ConsultationAgent (ASR + SOAP) --> HeARAgent (Respiratory Analysis) --> FHIR Upload --> Firestore Persistence

**Agents used:**
- `ConsultationAgent` — Audio transcription (Whisper) + SOAP note generation (MedGemma)
- `HeARAgent` — Respiratory sound analysis using Google HeAR model
- `ClinicalOrchestrator` — Coordinates agents and manages encounter state

**Prerequisites:** Run `00_setup_and_models.ipynb` first to load models and initialize the orchestrator.

## 0. Colab Bootstrap (run this first)

Auto-detects environment. In Colab, clones the private repo using your GitHub PAT.

**One-time setup:** In Colab, go to the **Key icon** in the left sidebar > Add a secret named `GITHUB_PAT` with your [GitHub Personal Access Token](https://github.com/settings/tokens).

In [ ]:
# ===========================================================
# Colab Bootstrap - run this cell first (works locally too)
# ===========================================================
import os, sys

try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_DIR = '/content/omnicare-clinical-copilot'

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        token = userdata.get('GITHUB_PAT')
        repo_url = f'https://{token}@github.com/Yashground/omnicare-clinical-copilot.git'
        os.system(f'git clone {repo_url} {REPO_DIR}')
    NOTEBOOKS_DIR = os.path.join(REPO_DIR, 'notebooks')
    os.makedirs('/content/encounters', exist_ok=True)
    os.makedirs('/content/sample_data', exist_ok=True)
else:
    NOTEBOOKS_DIR = os.path.dirname(os.path.abspath('__file__'))

if NOTEBOOKS_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOKS_DIR)

print(f'Environment ready | Colab: {IN_COLAB} | Notebooks dir: {NOTEBOOKS_DIR}')

## 1. Select Patient Scenario

Choose a clinical scenario for dynamic encounter generation. Each scenario
includes demographics, conditions, medications, vitals, and a transcript seed.

Available scenarios are loaded from `patient_simulator.py` -- no more hardcoded data.

In [ ]:
from utils.patient_simulator import list_scenarios, get_scenario

# Display available scenarios
scenarios = list_scenarios()
print("Available clinical scenarios:\n")
for i, s in enumerate(scenarios):
    print(f"  [{i}] {s['id']:<25s} | {s['patient']:<20s} | {s['chief_complaint']}")

# ---------------------------------------------------------------
# SELECT SCENARIO HERE
# Options:
#   get_scenario("pneumonia_diabetic")   — specific scenario by ID
#   get_scenario("chest_pain_cardiac")
#   get_scenario("copd_exacerbation")
#   get_scenario("uti_elderly")
#   get_scenario("asthma_pediatric_adult")
#   get_scenario()                       — random selection
# ---------------------------------------------------------------
scenario = get_scenario()  # random selection

demo = scenario["demographics"]
print(f"\n{'='*60}")
print(f"SELECTED SCENARIO: {scenario['id']}")
print(f"{'='*60}")
print(f"Patient:         {demo['name']}")
print(f"Gender:          {demo['gender']}")
print(f"DOB:             {demo['dob']}")
print(f"MRN:             {demo['mrn']}")
print(f"Chief Complaint: {scenario['chief_complaint']}")
print(f"Conditions:      {', '.join(c['display'] for c in scenario['conditions'])}")
print(f"Medications:     {', '.join(m['display'] for m in scenario['medications'])}")
print(f"Allergies:       {scenario.get('allergies', 'NKDA')}")

## 2. Start Encounter

Create a new encounter in Firestore (with automatic fallback to local JSON)
using the ClinicalOrchestrator. The orchestrator populates demographics from
the selected scenario and assigns an encounter ID.

In [ ]:
# Verify orchestrator is available from Notebook 00
try:
    _ = orchestrator
    _ = models
    print(f"Orchestrator ready with agents: {list(orchestrator.agents.keys())}")
except NameError:
    raise RuntimeError(
        "orchestrator / models not found. Run Notebook 00 first to load models "
        "and initialize the ClinicalOrchestrator."
    )

# Start a new encounter with the selected scenario
encounter_id = orchestrator.start_encounter(scenario=scenario)

print(f"\nEncounter ID:  {encounter_id}")
print(f"Patient:       {scenario['demographics']['name']}")
print(f"MRN:           {scenario['demographics']['mrn']}")
print(f"Scenario:      {scenario['id']}")
print(f"Storage:       Firestore (with local JSON fallback)")

## 3. Audio Input

Choose one of three input methods:

| Option | Description | When to use |
|--------|-------------|-------------|
| **A: Record in browser** | Colab audio recording widget | Live demo in Colab |
| **B: Upload audio file** | Upload a WAV/MP3 from disk | Pre-recorded consultation audio |
| **C: Dynamic transcript** | MedGemma generates transcript from scenario | No audio available (default) |

Set `INPUT_MODE` below to `"record"`, `"upload"`, or `"dynamic"`.

In [ ]:
# ---------------------------------------------------------------
# SELECT INPUT MODE: "record", "upload", or "dynamic"
# ---------------------------------------------------------------
INPUT_MODE = "dynamic"

audio_path = None
transcript = None
use_dynamic = False

if INPUT_MODE == "record":
    # === Option A: Record audio in Colab browser ===
    from IPython.display import HTML, Audio, display
    from google.colab import output
    from base64 import b64decode
    import numpy as np

    RECORD_HTML = """
    <script>
    let recorder, chunks = [];
    async function startRecording() {
      chunks = [];
      const stream = await navigator.mediaDevices.getUserMedia(
        {audio: {sampleRate: 16000, channelCount: 1}}
      );
      recorder = new MediaRecorder(stream);
      recorder.ondataavailable = e => chunks.push(e.data);
      recorder.start();
      document.getElementById('rec_status').innerText = 'Recording... click Stop when done.';
      document.getElementById('rec_status').style.color = 'red';
    }
    async function stopRecording() {
      recorder.stop();
      recorder.onstop = async () => {
        const blob = new Blob(chunks, {type: 'audio/webm'});
        const reader = new FileReader();
        reader.onload = () => {
          google.colab.kernel.invokeFunction('notebook.save_recording', [reader.result], {});
          document.getElementById('rec_status').innerText = 'Saved!';
          document.getElementById('rec_status').style.color = 'green';
        };
        reader.readAsDataURL(blob);
      };
    }
    </script>
    <button onclick="startRecording()" style="padding:8px 16px;font-size:14px;">
      Start Recording</button>
    <button onclick="stopRecording()" style="padding:8px 16px;font-size:14px;margin-left:8px;">
      Stop Recording</button>
    <p id="rec_status" style="margin-top:8px;">Ready — click Start to begin.</p>
    """

    _recorded_audio_path = "/content/recorded_consultation.wav"

    def _save_recording(data_url):
        """Callback invoked by the JS recording widget."""
        import soundfile as sf
        header, b64data = data_url.split(",", 1)
        raw_bytes = b64decode(b64data)
        # Write raw webm, then convert via librosa to 16kHz WAV
        webm_path = "/content/_temp_recording.webm"
        with open(webm_path, "wb") as f:
            f.write(raw_bytes)
        import librosa
        audio_arr, sr = librosa.load(webm_path, sr=16000, mono=True)
        sf.write(_recorded_audio_path, audio_arr, 16000)
        print(f"Recording saved: {_recorded_audio_path} ({len(audio_arr)/16000:.1f}s)")

    output.register_callback("notebook.save_recording", _save_recording)
    display(HTML(RECORD_HTML))
    print("Record your consultation audio above, then run the next cell.")
    print("After recording, set: audio_path = '/content/recorded_consultation.wav'")

elif INPUT_MODE == "upload":
    # === Option B: Upload audio file ===
    from google.colab import files
    print("Upload a WAV or MP3 file (16kHz mono recommended):")
    uploaded = files.upload()
    if uploaded:
        audio_path = list(uploaded.keys())[0]
        print(f"Uploaded: {audio_path}")
    else:
        print("No file uploaded. Falling back to dynamic transcript generation.")
        use_dynamic = True

elif INPUT_MODE == "dynamic":
    # === Option C: Dynamic transcript generation via MedGemma ===
    use_dynamic = True
    print("Using dynamic transcript generation from scenario via MedGemma.")
    print(f"Scenario seed preview: {scenario['transcript_seed'][:120]}...")

else:
    raise ValueError(f"Unknown INPUT_MODE: {INPUT_MODE!r}. Use 'record', 'upload', or 'dynamic'.")

print(f"\nInput configuration:")
print(f"  Mode:        {INPUT_MODE}")
print(f"  audio_path:  {audio_path}")
print(f"  use_dynamic: {use_dynamic}")

In [ ]:
# If you used INPUT_MODE="record", set the audio path after recording:
# audio_path = "/content/recorded_consultation.wav"

# If you used INPUT_MODE="upload" and want to override the path:
# audio_path = "your_uploaded_file.wav"

# Verify audio file exists (if using audio input)
if audio_path:
    import os
    if os.path.exists(audio_path):
        import librosa
        duration = librosa.get_duration(filename=audio_path)
        print(f"Audio file ready: {audio_path} ({duration:.1f}s)")
    else:
        print(f"WARNING: Audio file not found: {audio_path}")
        print("Falling back to dynamic transcript generation.")
        audio_path = None
        use_dynamic = True
else:
    print("No audio file — will use dynamic transcript generation.")

## 4. Run Consultation Agent

The orchestrator's `run_consultation()` method delegates to:
1. **ConsultationAgent** — Transcribes audio (Whisper ASR) or generates a dynamic transcript (MedGemma), then produces a structured SOAP note.
2. **HeARAgent** — Automatically runs if audio is provided, analyzing it for cough and respiratory events.

Both agents persist their results to Firestore.

In [ ]:
import time

print(f"Running consultation for encounter: {encounter_id}")
print(f"Input: {'Audio file: ' + audio_path if audio_path else 'Dynamic transcript from scenario'}")
start_time = time.time()

# The orchestrator handles:
#   - ConsultationAgent: ASR (if audio) or dynamic transcript, then SOAP generation
#   - HeARAgent: Respiratory analysis (if audio provided)
consultation_result = orchestrator.run_consultation(
    audio_path=audio_path,
    transcript=transcript,
    use_dynamic=use_dynamic,
)

elapsed = time.time() - start_time
print(f"\nConsultation completed in {elapsed:.1f}s")
print(f"Transcript length: {len(consultation_result['transcript'])} characters")

## 5. HeAR Respiratory Analysis

If audio was provided, the HeARAgent has already analyzed it for cough and
respiratory events. This section displays the results.

If no audio was used, HeAR analysis is skipped (embeddings require raw audio waveforms).

In [ ]:
hear_result = orchestrator.get_results().get("hear", None)

if hear_result:
    print("=" * 60)
    print("HeAR RESPIRATORY ANALYSIS RESULTS")
    print("=" * 60)
    print(f"\nSummary: {hear_result['hear_summary']}")

    events = hear_result.get("hear_events", [])
    if events:
        print(f"\nDetected {len(events)} respiratory event(s):\n")
        for i, evt in enumerate(events, 1):
            print(f"  [{i}] {evt['event_type']:<20s} "
                  f"at {evt['start_sec']:.1f}s - {evt['end_sec']:.1f}s "
                  f"(confidence: {evt['confidence']:.0%})")

        suggestion = hear_result.get("clinical_suggestion", "")
        if suggestion:
            print(f"\n--- Clinical Suggestion (MedGemma) ---")
            print(suggestion)
    else:
        print("\nNo respiratory events detected in the audio.")
else:
    print("HeAR analysis was not performed (no audio input provided).")
    print("To enable HeAR analysis, use INPUT_MODE='record' or 'upload' in Section 3.")

## 6. Display SOAP Note

The ConsultationAgent parsed the raw MedGemma output into structured SOAP sections.
Each section is displayed below with its content and character count.

In [ ]:
soap_sections = consultation_result["soap_note"]
soap_raw = consultation_result.get("soap_raw", "")

# Display the full transcript
print("=" * 60)
print("CONSULTATION TRANSCRIPT")
print("=" * 60)
print(consultation_result["transcript"])

# Display parsed SOAP sections
print("\n" + "=" * 60)
print("SOAP NOTE (Parsed Sections)")
print("=" * 60)

section_labels = {
    "subjective": "Subjective",
    "objective": "Objective",
    "assessment": "Assessment",
    "plan": "Plan",
}

for key, label in section_labels.items():
    content = soap_sections.get(key, "")
    status = f"{len(content)} chars" if content.strip() else "EMPTY"
    print(f"\n--- {label} ({status}) ---")
    if content.strip():
        print(content)
    else:
        print("  (No content extracted for this section)")

# Summary
total_chars = sum(len(v) for v in soap_sections.values())
filled = sum(1 for v in soap_sections.values() if v.strip())
print(f"\n{'='*60}")
print(f"SOAP completeness: {filled}/4 sections filled, {total_chars} total characters")

## 7. Upload Patient Data to FHIR Store

Generate a FHIR R4 Bundle from the selected scenario (Patient, Observations,
Conditions, MedicationRequests, AllergyIntolerance) and upload it to the
Google Cloud Healthcare API FHIR store.

This makes the patient's structured data available for querying in later notebooks
(admission vitals, discharge, etc.).

In [ ]:
import json
from utils.patient_simulator import generate_fhir_bundle

# Generate FHIR Bundle from the scenario
fhir_bundle = generate_fhir_bundle(scenario)

resource_types = {}
for entry in fhir_bundle.get("entry", []):
    rt = entry["resource"]["resourceType"]
    resource_types[rt] = resource_types.get(rt, 0) + 1

print("Generated FHIR R4 Bundle:")
print(f"  Bundle type: {fhir_bundle['type']}")
print(f"  Total entries: {len(fhir_bundle['entry'])}")
for rt, count in resource_types.items():
    print(f"    - {rt}: {count}")

# Upload to Healthcare API FHIR store
print(f"\nUploading to Healthcare API FHIR store...")
try:
    from utils.healthcare_api import upload_fhir_bundle
    result = upload_fhir_bundle(fhir_bundle)
    uploaded_count = len(result.get("entry", []))
    print(f"Upload successful: {uploaded_count} resources created/updated in FHIR store.")
except Exception as e:
    print(f"FHIR store upload failed: {e}")
    print("This is expected if Healthcare API is not configured.")
    print("The bundle is still available locally for use in subsequent notebooks.")

    # Save locally as fallback
    import os, tempfile
    local_path = os.path.join(tempfile.gettempdir(), f"fhir_bundle_{encounter_id}.json")
    with open(local_path, "w") as f:
        json.dump(fhir_bundle, f, indent=2)
    print(f"Bundle saved locally: {local_path}")

## 8. View Full Encounter (from Firestore)

Load the complete encounter document from Firestore (or local JSON fallback)
to verify all consultation data was persisted correctly.

This also shows the agent activity logs if Firestore is available.

In [ ]:
import json
from utils.encounter_state import load_encounter

# Load the full encounter
enc = load_encounter(encounter_id)

# Display encounter summary
print("=" * 60)
print(f"ENCOUNTER SUMMARY: {encounter_id}")
print("=" * 60)
print(f"Patient:     {enc.get('patient', {}).get('name', 'N/A')}")
print(f"MRN:         {enc.get('patient', {}).get('mrn', 'N/A')}")
print(f"Status:      {enc.get('status', 'N/A')}")
print(f"Created:     {enc.get('created_at', 'N/A')}")
print(f"Updated:     {enc.get('updated_at', 'N/A')}")

# Show which stages have data
stages = enc.get("stages", {})
print(f"\nStage completion:")
for stage_name, stage_data in stages.items():
    has_data = stage_data.get("timestamp") is not None
    indicator = "[x]" if has_data else "[ ]"
    print(f"  {indicator} {stage_name}")

# Show consultation stage details
consultation = stages.get("consultation", {})
if consultation.get("transcript"):
    print(f"\nConsultation transcript: {len(consultation['transcript'])} chars")
soap = consultation.get("soap_note", {})
if isinstance(soap, dict):
    filled = sum(1 for v in soap.values() if isinstance(v, str) and v.strip())
    print(f"SOAP sections filled: {filled}/4")
hear = consultation.get("hear_findings", [])
if hear:
    print(f"HeAR findings: {len(hear)} respiratory events")

# Show agent activity logs (Firestore only)
logs = orchestrator.get_agent_logs()
if logs:
    print(f"\n--- Agent Activity Log ({len(logs)} entries) ---")
    for log in logs:
        print(f"  [{log.get('agent', '?')}] {log.get('action', '?')}: "
              f"{log.get('details', '')[:80]}")

# Full JSON (truncated for display)
print(f"\n--- Full Encounter JSON (first 3000 chars) ---")
print(json.dumps(enc, indent=2, default=str)[:3000])

# Pass-forward info for next notebook
print(f"\n{'='*60}")
print(f"Consultation phase complete!")
print(f"Next: Run 02_admission_vitals_fhir.ipynb")
print(f"  encounter_id = '{encounter_id}'")
print(f"  scenario_id  = '{scenario['id']}'")